In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

In [ ]:
# -*- coding: utf-8 -*-
"""Bước 1: tính định lý Bayes bằng tay, không cần thư viện học máy.

Chú thích viết tiếng Việt có dấu, phần in ra màn hình viết không dấu.
"""


def bayes(tien_nghiem, kha_nang, kha_nang_nguoc):
    """Tra ve xac suat hau nghiem.

    tien_nghiem : P(A), niem tin truoc khi co bang chung
    kha_nang : P(B | A), bang chung xuat hien khi A dung
    kha_nang_nguoc : P(B | khong A), bang chung xuat hien khi A sai
    """
    tu_so = kha_nang * tien_nghiem
    mau_so = tu_so + kha_nang_nguoc * (1 - tien_nghiem)
    return tu_so / mau_so


print("Vi du 1: xet nghiem benh")
print(" Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.")
p = bayes(tien_nghiem=0.01, kha_nang=0.99, kha_nang_nguoc=0.01)
print(f" Duong tinh thi kha nang co benh that = {p:.4f}")
print()

# Giữ nguyên chất lượng xét nghiệm, chỉ đổi tỷ lệ mắc bệnh
print("Cung xet nghiem do, doi ty le mac benh:")
print(" Ty le mac benh Hau nghiem khi duong tinh")
for ty_le in [0.001, 0.01, 0.05, 0.10, 0.50]:
    print(f" {ty_le:6.3f} {bayes(ty_le, 0.99, 0.01):.4f}")
print()

print("Vi du 2: hai day chuyen san xuat")
print(" Day A lam 80 phan tram san pham, ty le loi 1 phan tram.")
print(" Day B lam 20 phan tram san pham, ty le loi 5 phan tram.")
# Ở đây "A đúng" nghĩa là sản phẩm đến từ dây B
p_b = bayes(tien_nghiem=0.20, kha_nang=0.05, kha_nang_nguoc=0.01)
print(f" Nhat duoc san pham loi, kha nang no tu day B = {p_b:.4f}")
print()

print("Kiem lai bang cach dem tren 1000 san pham:")
loi_a = 800 * 0.01
loi_b = 200 * 0.05
print(f" Day A: 800 san pham, {loi_a:.0f} cai loi")
print(f" Day B: 200 san pham, {loi_b:.0f} cai loi")
tong_loi = loi_a + loi_b
print(f" Ty le tu day B = {loi_b} / {tong_loi} = {loi_b / tong_loi:.4f}")

Vi du 1: xet nghiem benh
 Ty le mac benh 1 phan tram, xet nghiem dung 99 phan tram.
 Duong tinh thi kha nang co benh that = 0.5000

Cung xet nghiem do, doi ty le mac benh:
 Ty le mac benh Hau nghiem khi duong tinh
  0.001 0.0902
  0.010 0.5000
  0.050 0.8390
  0.100 0.9167
  0.500 0.9900

Vi du 2: hai day chuyen san xuat
 Day A lam 80 phan tram san pham, ty le loi 1 phan tram.
 Day B lam 20 phan tram san pham, ty le loi 5 phan tram.
 Nhat duoc san pham loi, kha nang no tu day B = 0.5556

Kiem lai bang cach dem tren 1000 san pham:
 Day A: 800 san pham, 8 cai loi
 Day B: 200 san pham, 10 cai loi
 Ty le tu day B = 10.0 / 18.0 = 0.5556


In [ ]:
# -*- coding: utf-8 -*-
"""Bước 2: đọc kho tin nhắn và biến văn bản thành bảng số."""


df = pd.read_csv("tin_nhan.csv")

print("Kich thuoc bang:", df.shape)
print()
print("So tin theo nhan (1 la rac, 0 la thuong):")
print(df["la_rac"].value_counts())
print()
print("Ty le tin rac:", round(df["la_rac"].mean(), 4))
print()

print("Ba tin nhan dau tien:")
for i in range(3):
    print(f" [{df.la_rac[i]}] {df.noi_dung[i]}")
print()

# token_pattern mặc định của scikit-learn bỏ mất từ một ký tự,
# nên ta chỉ định tách từ bằng khoảng trắng cho đơn giản.
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])

print("Kich thuoc bang dem tu:", X.shape)
print("So tu trong tu dien:", len(vec.vocabulary_))
print()

# Đếm tổng số lần xuất hiện của từng từ, tách theo hai lớp
dem = pd.DataFrame(X.toarray(), columns=vec.get_feature_names_out())
dem["la_rac"] = df["la_rac"]
tong = dem.groupby("la_rac").sum().T
tong.columns = ["thuong", "rac"]

print("Muoi tu xuat hien nhieu nhat trong tin rac:")
print(tong.sort_values("rac", ascending=False).head(10))
print()
print("Muoi tu xuat hien nhieu nhat trong tin thuong:")
print(tong.sort_values("thuong", ascending=False).head(10))

Kich thuoc bang: (300, 2)

So tin theo nhan (1 la rac, 0 la thuong):
la_rac
0    180
1    120
Name: count, dtype: int64

Ty le tin rac: 0.4

Ba tin nhan dau tien:
 [0] cáo gửi cáo thầy nhóm cáo gửi có cho
 [1] miễn giảm ngay hạn miễn tay nhé thưởng cho nhanh
 [0] cho bạn mai gửi nhóm bài chiều

Kich thuoc bang dem tu: (300, 40)
So tu trong tu dien: 40

Muoi tu xuat hien nhieu nhat trong tin rac:
       thuong  rac
tay         3   48
ngay        4   47
nhanh       9   44
chót        6   43
giảm        6   42
lúc        39   42
quà         8   41
click       5   40
phí         6   40
miễn        4   39

Muoi tu xuat hien nhieu nhat trong tin thuong:
       thuong  rac
thầy       74    7
bài        73    4
cáo        72    8
chiều      68    8
ạ          65    7
tài        64   11
tập        63    6
gửi        61   10
lớp        60    2
nhóm       58    7


In [ ]:
# -*- coding: utf-8 -*-
"""Bước 3: huấn luyện bộ lọc thư rác và chấm điểm nó."""

import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB

df = pd.read_csv("tin_nhan.csv")
vec = CountVectorizer(token_pattern=r"(?u)\S+")
X = vec.fit_transform(df["noi_dung"])
y = df["la_rac"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)
print("So tin de hoc :", X_train.shape[0])
print("So tin de kiem tra:", X_test.shape[0])
print()

mo_hinh = MultinomialNB(alpha=1.0)
mo_hinh.fit(X_train, y_train)
y_pred = mo_hinh.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print("Ma tran nham lan")
print(f" TN = {tn:2d} thu thuong duoc giu dung")
print(f" FP = {fp:2d} thu thuong bi xoa nham")
print(f" FN = {fn:2d} thu rac lot luoi")
print(f" TP = {tp:2d} thu rac bi chan dung")
print()

print(f"Accuracy = {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision = {precision_score(y_test, y_pred):.4f}")
print(f"Recall = {recall_score(y_test, y_pred):.4f}")
print()

# Thử bộ lọc trên vài tin nhắn tự gõ
tin_moi = [
    "trung thuong khuyen mai nhanh tay click ngay",
    "hop nhom bao cao chieu mai phong 305",
    "ban gui giup em tai lieu bai tap nhe",
]
X_moi = vec.transform(tin_moi)
p_rac = mo_hinh.predict_proba(X_moi)[:, 1]

print("Thu tren ba tin nhan tu go:")
for tin, p in zip(tin_moi, p_rac):
    nhan = "RAC" if p >= 0.5 else "thuong"
    print(f" [{nhan:6s}] p = {p:.4f} <- {tin}")

So tin de hoc : 225
So tin de kiem tra: 75

Ma tran nham lan
 TN = 45 thu thuong duoc giu dung
 FP =  0 thu thuong bi xoa nham
 FN =  4 thu rac lot luoi
 TP = 26 thu rac bi chan dung

Accuracy = 0.9467
Precision = 1.0000
Recall = 0.8667

Thu tren ba tin nhan tu go:
 [RAC   ] p = 0.9992 <- trung thuong khuyen mai nhanh tay click ngay
 [thuong] p = 0.0983 <- hop nhom bao cao chieu mai phong 305
 [thuong] p = 0.0719 <- ban gui giup em tai lieu bai tap nhe


In [ ]:
# -*- coding: utf-8 -*-
"""Bước 4: tự cài Naive Bayes trên một kho nhỏ để thấy vấn đề số không.

Đoạn này cố ý KHÔNG dùng scikit-learn, vì thư viện đã làm trơn sẵn nên
sinh viên không nhìn thấy chỗ hỏng.
"""

from collections import Counter

# Kho tin nhắn tí hon, viết thẳng vào mã cho dễ theo dõi
kho = [
    ("trung thuong ngay", 1),
    ("khuyen mai giam gia", 1),
    ("trung thuong click", 1),
    ("hop nhom chieu mai", 0),
    ("bai tap nhom nop mai", 0),
    ("thay gui tai lieu", 0),
]

tu_dien = sorted({t for cau, _ in kho for t in cau.split()})
V = len(tu_dien)
print(f"Tu dien co {V} tu:")
# In thành nhiều dòng, mỗi dòng 9 từ, để không tràn ra ngoài khung
for i in range(0, V, 9):
    print(" " + " ".join(tu_dien[i : i + 9]))
print()

# Đếm số lần mỗi từ xuất hiện trong từng lớp
dem = {0: Counter(), 1: Counter()}
tong = {0: 0, 1: 0}
for cau, nhan in kho:
    for t in cau.split():
        dem[nhan][t] += 1
        tong[nhan] += 1

print("So lan xuat hien cua tung tu")
print(f" {'tu':12s} {'rac':>5s} {'thuong':>7s}")
for t in tu_dien:
    print(f" {t:12s} {dem[1][t]:5d} {dem[0][t]:7d}")
print()


def xac_suat(tu, nhan, alpha):
    """P(tu | nhan) co lam tron Laplace voi he so alpha."""
    return (dem[nhan][tu] + alpha) / (tong[nhan] + alpha * V)


def diem(cau, nhan, alpha):
    """Tich cua xac suat tien nghiem va cac xac suat tu."""
    p = 0.5  # hai lop can bang nen tien nghiem bang nhau
    for t in cau.split():
        if t in tu_dien:
            p *= xac_suat(t, nhan, alpha)
    return p


cau_thu = "trung thuong bai tap"
print(f"Tin nhan thu: \"{cau_thu}\"")
print("Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.")
print()

for alpha in [0.0, 1.0]:
    d_rac = diem(cau_thu, 1, alpha)
    d_thuong = diem(cau_thu, 0, alpha)
    ket = "rac" if d_rac > d_thuong else "thuong"
    print(f"alpha = {alpha}")
    print(f"  diem lop rac = {d_rac:.10f}")
    print(f"  diem lop thuong = {d_thuong:.10f}")
    print(f"  ket luan = {ket}")
    print()

print("Khong lam tron, CA HAI diem deu bang 0 nen mo hinh te liet.")
print("Lop rac bang 0 vi tu 'bai' chua tung xuat hien o lop rac.")
print("Lop thuong bang 0 vi tu 'trung' chua tung xuat hien o lop thuong.")
print("Co lam tron, diem lop rac lon hon va mo hinh ket luan dung.")

Tu dien co 18 tu:
 bai chieu click gia giam gui hop khuyen lieu
 mai ngay nhom nop tai tap thay thuong trung

So lan xuat hien cua tung tu
 tu             rac  thuong
 bai              0       1
 chieu            0       1
 click            1       0
 gia              1       0
 giam             1       0
 gui              0       1
 hop              0       1
 khuyen           1       0
 lieu             0       1
 mai              1       2
 ngay             1       0
 nhom             0       2
 nop              0       1
 tai              0       1
 tap              0       1
 thay             0       1
 thuong           2       0
 trung            2       0

Tin nhan thu: "trung thuong bai tap"
Tu 'bai' chi xuat hien o lop thuong, chua bao gio o lop rac.

alpha = 0.0
  diem lop rac = 0.0000000000
  diem lop thuong = 0.0000000000
  ket luan = thuong

alpha = 1.0
  diem lop rac = 0.0000073212
  diem lop thuong = 0.0000021656
  ket luan = rac

Khong lam tron, CA HAI diem deu bang 0 n

In [ ]:
# -*- coding: utf-8 -*-
"""Bước 5: Naive Bayes cho số liệu liên tục, đặt cạnh hồi quy logistic."""

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB

df = pd.read_csv("sinh_vien.csv")
X = df[["gio_on", "diem_giua_ky"]]
y = df["qua_mon"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

# Hai mô hình, cùng dữ liệu, cùng cách chia, nên so sánh mới công bằng
for ten, mo_hinh in [
    ("Naive Bayes ", GaussianNB()),
    ("Hoi quy logistic", LogisticRegression()),
]:
    mo_hinh.fit(X_train, y_train)
    y_pred = mo_hinh.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"{ten} Accuracy = {accuracy:.4f}")

Naive Bayes  Accuracy = 0.9000
Hoi quy logistic Accuracy = 0.9000


Bài tập

Bài 1

In [ ]:
#1. Khai bao xac suat tien nghiem cua ba nhom
p_hoc_deu = 0.50
p_hoc_don = 0.30
p_it_hoc = 0.20

#2. Xac suat qua mon cua tung nhom
p_qua_deu = 0.95
p_qua_don = 0.70
p_qua_it = 0.30

#3. Tinh tu so theo cong thuc Bayes
tu_so_deu = p_hoc_deu * p_qua_deu
tu_so_don = p_hoc_don * p_qua_don
tu_so_it = p_it_hoc * p_qua_it

#4. Tinh xac suat tong da qua mon
p_qua_mon = tu_so_deu + tu_so_don + tu_so_it

#5. Tinh xac suat hau nghiem
p_deu = tu_so_deu / p_qua_mon
p_don = tu_so_don / p_qua_mon
p_it = tu_so_it / p_qua_mon

print("XAC SUAT THUOC TUNG NHOM KHI DA QUA MON")
print(f"Nhom hoc deu: {p_deu:.4f}")
print(f"Nhom hoc don: {p_don:.4f}")
print(f"Nhom it hoc : {p_it:.4f}")

#6. Kiem tra tong xac suat
tong_xac_suat = p_deu + p_don + p_it

print(f"\nTong ba xac suat: {tong_xac_suat:.4f}")

XAC SUAT THUOC TUNG NHOM KHI DA QUA MON
Nhom hoc deu: 0.6376
Nhom hoc don: 0.2819
Nhom it hoc : 0.0805

Tong ba xac suat: 1.0000


Bài 2

In [ ]:
#1. Chuyen noi dung tin nhan thanh bang dem tu
try:
    van_ban = df_tin["noi_dung"].fillna("").astype(str)
    nhan = df_tin["la_rac"].astype(int)

    vectorizer_b2 = CountVectorizer(token_pattern=r"(?u)\S+")
    X_b2 = vectorizer_b2.fit_transform(van_ban)

    #2. Huan luyen tren toan bo 300 tin nhan
    mo_hinh_b2 = MultinomialNB(alpha=1.0)
    mo_hinh_b2.fit(X_b2, nhan)

    #3. Tinh hieu log xac suat cua tung tu
    # Hang 0 la tin thuong, hang 1 la tin rac
    chenhlech = (
        mo_hinh_b2.feature_log_prob_[1]
        - mo_hinh_b2.feature_log_prob_[0]
    )

    danh_sach_tu = np.array(
        vectorizer_b2.get_feature_names_out()
    )

    #4. Lay 5 tu nghieng manh nhat ve tin rac
    vi_tri_rac = np.argsort(chenhlech)[-5:][::-1]

    print("5 TU TO CAO TIN RAC MANH NHAT")
    for i in vi_tri_rac:
        print(f"{danh_sach_tu[i]:20s} {chenhlech[i]:.4f}")

    #5. Lay 5 tu nghieng manh nhat ve tin thuong
    vi_tri_thuong = np.argsort(chenhlech)[:5]

    print("\n5 TU CHO THAY TIN THUONG MANH NHAT")
    for i in vi_tri_thuong:
        print(f"{danh_sach_tu[i]:20s} {chenhlech[i]:.4f}")

except Exception as e:
    print("Loi:", type(e).__name__, e)

5 TU TO CAO TIN RAC MANH NHAT
mãi                  3.2034
tay                  2.9056
ngay                 2.6618
miễn                 2.4795
trúng                2.3742

5 TU CHO THAY TIN THUONG MANH NHAT
lớp                  -2.6122
em                   -2.5618
bài                  -2.2946
báo                  -2.2741
phòng                -2.2741


Bài 3

In [ ]:
#1. Tao bang dem tu cho bo du lieu
try:
    van_ban_b3 = df_tin["noi_dung"].fillna("").astype(str)
    nhan_b3 = df_tin["la_rac"].astype(int)

    vectorizer_b3 = CountVectorizer(token_pattern=r"(?u)\S+")
    X_b3 = vectorizer_b3.fit_transform(van_ban_b3)

    #2. Chia du lieu thanh tap huan luyen va tap kiem tra
    X_train_b3, X_test_b3, y_train_b3, y_test_b3 = train_test_split(
        X_b3,
        nhan_b3,
        test_size=0.25,
        random_state=17,
        stratify=nhan_b3
    )

    #3. Thu nghiem voi cac gia tri alpha
    danh_sach_alpha = [0.001, 0.1, 1.0, 10.0, 100.0]

    print(f"{'Alpha':>10s} {'Accuracy':>12s} "
          f"{'Precision':>12s} {'Recall':>12s}")

    for alpha in danh_sach_alpha:
        mo_hinh_b3 = MultinomialNB(alpha=alpha)
        mo_hinh_b3.fit(X_train_b3, y_train_b3)

        du_doan_b3 = mo_hinh_b3.predict(X_test_b3)

        accuracy = accuracy_score(y_test_b3, du_doan_b3)
        precision = precision_score(
            y_test_b3, du_doan_b3, zero_division=0
        )
        recall = recall_score(
            y_test_b3, du_doan_b3, zero_division=0
        )

        print(f"{alpha:10.3f} {accuracy:12.4f} "
              f"{precision:12.4f} {recall:12.4f}")

except Exception as e:
    print("Loi:", type(e).__name__, e)

     Alpha     Accuracy    Precision       Recall
     0.001       0.9467       1.0000       0.8667
     0.100       0.9467       1.0000       0.8667
     1.000       0.9467       1.0000       0.8667
    10.000       0.9467       1.0000       0.8667
   100.000       0.9467       1.0000       0.8667


Alpha quá lớn làm xác suất giữa các từ ít khác biệt, khiến mô hình khó phân biệt tin rác và tin thường, từ đó giảm hiệu quả dự đoán.

Bài 4

In [ ]:
#1. Chuan bi du lieu va chia tap train/test
try:
    van_ban_b4 = (
        df_tin["noi_dung"].fillna("").astype(str).tolist()
    )
    nhan_b4 = df_tin["la_rac"].astype(int).reset_index(drop=True)

    chi_so = np.arange(len(van_ban_b4))

    train_idx, test_idx = train_test_split(
        chi_so,
        test_size=0.25,
        random_state=17,
        stratify=nhan_b4
    )

    #2. Tach tu bang khoang trang va chuyen thanh chu thuong
    cac_tu = [cau.lower().split() for cau in van_ban_b4]

    # Tao tu dien tu toan bo 300 tin nhan de so sanh cong bang
    tu_dien_b4 = sorted({
        tu for danh_sach in cac_tu for tu in danh_sach
    })

    V = len(tu_dien_b4)
    alpha_b4 = 1.0

    print("So tin nhan:", len(van_ban_b4))
    print("So tu trong tu dien:", V)
    print("So tin huan luyen:", len(train_idx))
    print("So tin kiem tra:", len(test_idx))

    #3. Tu dem so tu cua tung lop tren tap huan luyen
    dem_tu_b4 = {
        0: Counter(),
        1: Counter()
    }

    tong_tu_b4 = {
        0: 0,
        1: 0
    }

    so_mau_b4 = Counter()

    for i in train_idx:
        lop = int(nhan_b4.iloc[i])
        so_mau_b4[lop] += 1

        for tu in cac_tu[i]:
            dem_tu_b4[lop][tu] += 1
            tong_tu_b4[lop] += 1

    #4. Tu tinh log xac suat voi lam tron Laplace
    log_xac_suat_b4 = {}

    for lop in [0, 1]:
        mau_so = tong_tu_b4[lop] + alpha_b4 * V
        log_xac_suat_b4[lop] = {}

        for tu in tu_dien_b4:
            tu_so = dem_tu_b4[lop][tu] + alpha_b4
            log_xac_suat_b4[lop][tu] = math.log(tu_so / mau_so)

    #5. Tinh log xac suat tien nghiem cua tung lop
    log_tien_nghiem_b4 = {}

    for lop in [0, 1]:
        log_tien_nghiem_b4[lop] = math.log(
            so_mau_b4[lop] / len(train_idx)
        )

    #6. Tu viet ham du doan bang tong log xac suat
    def du_doan_b4(cau):
        cac_tu_trong_cau = cau.lower().split()
        diem_lop = {}

        for lop in [0, 1]:
            diem = log_tien_nghiem_b4[lop]

            for tu in cac_tu_trong_cau:
                if tu in log_xac_suat_b4[lop]:
                    diem += log_xac_suat_b4[lop][tu]

            diem_lop[lop] = diem

        return max(diem_lop, key=diem_lop.get)

    #7. Du doan tren tap kiem tra bang bo loc tu viet
    y_test_b4 = nhan_b4.iloc[test_idx].to_numpy()

    du_doan_tu_viet = np.array([
        du_doan_b4(van_ban_b4[i])
        for i in test_idx
    ])

    accuracy_tu_viet = accuracy_score(
        y_test_b4, du_doan_tu_viet
    )

    #8. Huan luyen MultinomialNB de doi chieu
    vectorizer_b4 = CountVectorizer(token_pattern=r"(?u)\S+")
    X_b4 = vectorizer_b4.fit_transform(van_ban_b4)

    mo_hinh_b4 = MultinomialNB(alpha=alpha_b4)
    mo_hinh_b4.fit(
        X_b4[train_idx],
        nhan_b4.iloc[train_idx]
    )

    du_doan_thu_vien = mo_hinh_b4.predict(X_b4[test_idx])

    accuracy_thu_vien = accuracy_score(
        y_test_b4, du_doan_thu_vien
    )

    #9. So sanh hai ket qua
    print("\nKET QUA SO SANH")
    print(f"Accuracy bo loc tu viet: {accuracy_tu_viet:.4f}")
    print(f"Accuracy MultinomialNB:  {accuracy_thu_vien:.4f}")

    so_mau_khac_nhau = np.sum(
        du_doan_tu_viet != du_doan_thu_vien
    )
    print("So du doan khac nhau:", so_mau_khac_nhau)

except Exception as e:
    print("Loi:", type(e).__name__, e)

So tin nhan: 300
So tu trong tu dien: 40
So tin huan luyen: 225
So tin kiem tra: 75

KET QUA SO SANH
Accuracy bo loc tu viet: 0.9467
Accuracy MultinomialNB:  0.9467
So du doan khac nhau: 0


Bộ lọc tự viết và MultinomialNB có cùng công thức và dữ liệu nên Accuracy phải gần bằng nhau. Nếu chênh lệch lớn, cần kiểm tra cách đếm từ, làm trơn, xác suất tiên nghiệm và tổng log xác suất để tránh sai số khi tính toán.

Bài 5

In [ ]:
#1. Chon hai bien dau vao va nhan
try:
    X_b5 = df_sv[["gio_on", "diem_giua_ky"]].astype(float)
    y_b5 = df_sv["qua_mon"].astype(int)

    print("Gia tri nho nhat cua cac bien:")
    print(X_b5.min())

    #2. Chia du lieu co dinh de so sanh cong bang
    X_train_b5, X_test_b5, y_train_b5, y_test_b5 = train_test_split(
        X_b5,
        y_b5,
        test_size=0.25,
        random_state=17,
        stratify=y_b5
    )

    #3. Thu MultinomialNB
    print("\nKET QUA MULTINOMIALNB")

    try:
        mo_hinh_multi_b5 = MultinomialNB(alpha=1.0)
        mo_hinh_multi_b5.fit(X_train_b5, y_train_b5)

        du_doan_multi_b5 = mo_hinh_multi_b5.predict(X_test_b5)

        acc_multi_b5 = accuracy_score(
            y_test_b5, du_doan_multi_b5
        )

        print(f"Accuracy = {acc_multi_b5:.4f}")

    except Exception as e:
        print("MultinomialNB bao loi:", type(e).__name__, e)

    #4. Thu GaussianNB tren cung tap du lieu
    print("\nKET QUA GAUSSIANNB")

    mo_hinh_gauss_b5 = GaussianNB()
    mo_hinh_gauss_b5.fit(X_train_b5, y_train_b5)

    du_doan_gauss_b5 = mo_hinh_gauss_b5.predict(X_test_b5)

    acc_gauss_b5 = accuracy_score(
        y_test_b5, du_doan_gauss_b5
    )

    print(f"Accuracy = {acc_gauss_b5:.4f}")

except Exception as e:
    print("Loi:", type(e).__name__, e)

Gia tri nho nhat cua cac bien:
gio_on          0.0
diem_giua_ky    0.3
dtype: float64

KET QUA MULTINOMIALNB
Accuracy = 0.8000

KET QUA GAUSSIANNB
Accuracy = 0.9000


Nếu hai cột đầu vào không có giá trị âm, MultinomialNB có thể chạy với dữ liệu không âm nhưng không phù hợp với biến liên tục. GaussianNB phù hợp hơn vì giả định dữ liệu mỗi lớp tuân theo phân phối Gaussian. Accuracy cần xác định từ kết quả chạy trên file `sinh_vien.csv`.

Bài 6

In [ ]:
#1. Lay toan bo tin thuong va 20 tin rac dau tien
try:
    tin_thuong_b6 = df_tin[df_tin["la_rac"] == 0]
    tin_rac_b6 = df_tin[df_tin["la_rac"] == 1].head(20)

    # Ghep thanh bo du lieu mat can bang
    df_mat_can_bang = pd.concat(
        [tin_thuong_b6, tin_rac_b6],
        ignore_index=True
    )

    print("SO TIN NHAN SAU KHI MAT CAN BANG")
    print(df_mat_can_bang["la_rac"].value_counts())
    print("Tong so tin:", len(df_mat_can_bang))

    #2. Tach du lieu dau vao va nhan
    van_ban_b6 = (
        df_mat_can_bang["noi_dung"].fillna("").astype(str)
    )
    nhan_b6 = df_mat_can_bang["la_rac"].astype(int)

    #3. Chuyen van ban thanh bang dem tu
    vectorizer_b6 = CountVectorizer(
        token_pattern=r"(?u)\S+"
    )
    X_b6 = vectorizer_b6.fit_transform(van_ban_b6)

    #4. Chia tap huan luyen va tap kiem tra
    X_train_b6, X_test_b6, y_train_b6, y_test_b6 = train_test_split(
        X_b6,
        nhan_b6,
        test_size=0.25,
        random_state=17,
        stratify=nhan_b6
    )

    #5. Huan luyen mo hinh
    mo_hinh_b6 = MultinomialNB(alpha=1.0)
    mo_hinh_b6.fit(X_train_b6, y_train_b6)

    #6. Du doan va tinh chi so danh gia
    du_doan_b6 = mo_hinh_b6.predict(X_test_b6)

    accuracy_b6 = accuracy_score(
        y_test_b6, du_doan_b6
    )

    recall_rac_b6 = recall_score(
        y_test_b6,
        du_doan_b6,
        pos_label=1,
        zero_division=0
    )

    print("\nKET QUA DANH GIA")
    print(f"Accuracy = {accuracy_b6:.4f}")
    print(f"Recall lop rac = {recall_rac_b6:.4f}")

except Exception as e:
    print("Loi:", type(e).__name__, e)

SO TIN NHAN SAU KHI MAT CAN BANG
la_rac
0    180
1     20
Name: count, dtype: int64
Tong so tin: 200

KET QUA DANH GIA
Accuracy = 0.9800
Recall lop rac = 0.8000


Khi dữ liệu mất cân bằng, Accuracy có thể cao dù mô hình bỏ sót nhiều tin rác. Vì vậy, cần đánh giá cả Accuracy và Recall của lớp rác để phản ánh chính xác hiệu quả bộ lọc.